In [1]:
import json
from collections import Counter

FILE_PATH = "/content/combined.jsonl"

def audit_dataset():
    stats = Counter()
    total_lines = 0

    REQUIRED_KEYS = ["instruction", "output"]
    # These are the tags we forced Gemma 3 27B to produce
    REQUIRED_BLUEPRINT_SECTIONS = [
        "[DIFFICULTY]", "[TOPICS]", "[GOAL]",
        "[LOGIC]", "[PSEUDOCODE]", "[CORNER CASES]"
    ]

    try:
        with open(FILE_PATH, 'r') as f:
            for line in f:
                total_lines += 1
                try:
                    data = json.loads(line)
                    output = data.get("output", "")

                    # 1. Structural Check
                    has_tags = "<tags>" in output and "</tags>" in output
                    has_think = "<think>" in output and "</think>" in output
                    has_code = "<code>" in output and "</code>" in output

                    # 2. Logic Check
                    missing_blueprint = [s for s in REQUIRED_BLUEPRINT_SECTIONS if s not in output]

                    if not (has_tags and has_think and has_code):
                        stats["Missing XML Tags (Fail)"] += 1
                    elif missing_blueprint:
                        stats[f"Incomplete Logic (Fail)"] += 1
                    else:
                        stats["Ready for Training (Pass) ✅"] += 1

                except json.JSONDecodeError:
                    stats["Corrupt JSON Line (Fail)"] += 1
    except FileNotFoundError:
        print("❌ Master file not found. Did you run the merge script?")
        return

    print("📊 STAGE 2 DATASET AUDIT REPORT")
    print("-" * 35)
    print(f"Total Lines Processed: {total_lines}")
    for category, count in stats.items():
        percentage = (count / total_lines) * 100
        print(f"{category}: {count} ({percentage:.1f}%)")

    if stats["Ready for Training (Pass) ✅"] < 1500:
        print("\n⚠️ WARNING: Your dataset is below 1,500 samples. You might need more epochs.")
    else:
        print("\n🚀 SUCCESS: Dataset size is optimal for 1.5B reasoning training.")

audit_dataset()

📊 STAGE 2 DATASET AUDIT REPORT
-----------------------------------
Total Lines Processed: 1971
Ready for Training (Pass) ✅: 1971 (100.0%)

🚀 SUCCESS: Dataset size is optimal for 1.5B reasoning training.


In [2]:
!pip install unsloth transformers datasets accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.0/447.0 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 124.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
from tqdm import tqdm

# --- 1. CORRECT PATHS ---
# Using the path from your Drive screenshot
model_path = "/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage1"
dataset_file = "/content/combined.jsonl"
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    # fix_mistral_regex = True # Removed, as it's not a valid argument for Qwen2 models
)

# --- 2. LORA REDUCTION (Prevents Mode Collapse) ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,            # Reduced from 32 to improve generalization
    lora_alpha = 32,   # Scaled down with rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout = 0,
    bias = "none",
)

# --- 3. FORMATTING WITH I/O EMPHASIS ---
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for i in tqdm(range(len(instructions)), desc="🛠️ Formatting"):
        # We add a hint to handle ALL test cases to break the "Single N" habit
        text = f"### Instruction (Handle all test cases using standard I/O):\n{instructions[i]}\n\n### Response:\n{outputs[i]}"
        texts.append(text)
    return { "text" : texts }

dataset = load_dataset("json", data_files = dataset_file, split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True, batch_size = 1000)

# --- 4. OPTIMIZED TRAINING (Lower Epochs) ---
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = 1.5,         # Reduced to prevent overfitting
        learning_rate = 1e-4,           # Slightly lower LR for stability
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs_refined",
    ),
)

# --- 5. EXECUTION ---
print("\n🚀 Starting Refined Stage 2 Training...")
trainer.train()

model.save_pretrained_merged("DeepSeek-R1-CP-Python-Stage2_Fixed", tokenizer, save_method = "merged_16bit")
!cp -r DeepSeek-R1-CP-Python-Stage2_Fixed /content/drive/MyDrive/SLM_DATA/

==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load /content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage1 as a legacy tokenizer.


/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage1 does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


...

Detected local model directory: /content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage1
No existing and accessible Hugging Face cache directory found.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:59<00:00, 59.35s/it]


Copied model.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:37<00:00, 97.11s/it]


Unsloth: Merge process complete. Saved to `/content/DeepSeek-R1-CP-Python-Stage2_Fixed`


In [4]:
import os

model_directory_to_check = "/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage1"

if os.path.exists(model_directory_to_check):
    print(f"Contents of '{model_directory_to_check}':")
    for item in os.listdir(model_directory_to_check):
        print(f"- {item}")
else:
    print(f"❌ The directory '{model_directory_to_check}' does not exist. Please ensure your Stage 1 model was saved correctly.")

Contents of '/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage1':
- chat_template.jinja
- special_tokens_map.json
- tokenizer.json
- tokenizer_config.json
- config.json
- .cache
- model.safetensors
- checksum.sha256


In [5]:
from unsloth import FastLanguageModel
import torch

# 1. Load your Fine-Tuned Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed", # 👈 Change to your actual checkpoint path
    max_seq_length = 4096,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

def test_model(question):
    prompt = f"### Instruction:\n{question}\n\n### Response:\n<tags>"

    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    # We force the model to start with <tags> and <think>
    outputs = model.generate(
        **inputs,
        max_new_tokens = 1500,
        use_cache = True,
        temperature = 0.1, # Keep it logical
        top_p = 0.9
    )

    result = tokenizer.batch_decode(outputs)[0]

    # Split the result to show you the stages
    print("\n🚀 MODEL REASONING & CODE:")
    print("-" * 40)
    print(result.split("### Response:\n")[-1])
    print("-" * 40)

# --- TEST IT WITH A NEW PROBLEM ---
new_problem = """
Problem: Given an array of integers, return the indices of the two numbers such that they add up to a specific target.
Input: nums = [2, 7, 11, 15], target = 9
Output: [0, 1]
"""

test_model(new_problem)

==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load /content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed as a legacy tokenizer.
--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in 


🚀 MODEL REASONING & CODE:
----------------------------------------
<tags> EASY, Hash Tables, Brute Force, Array Manipulation </tags>
<think>
```
[DIFFICULTY]: EASY
[TOPICS]: Hash Tables, Brute Force, Array Manipulation
[GOAL]: Find two distinct indices in the array whose values sum to a given target.
[LOGIC]: The core idea is to iterate through the array and use a hash map to store the difference between the target and the current element. If the difference exists in the hash map, return the indices of the current element and the stored difference. If not, add the current element and its index to the hash map.
[PSEUDOCODE]:
```python
def find_two_sum(nums, target):
    hash_map = {}
    for i, num in enumerate(nums):
        complement = target - num
        if complement in hash_map:
            return [hash_map[complement], i]
        hash_map[num] = i
    return []
```
[CORNER CASES]:
1. Empty array: Should return an empty list.
2. Array with no two numbers summing to the target: S

In [1]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.0/447.0 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224

In [8]:
import json
import random
from unsloth import FastLanguageModel
import torch

# --- 1. SETTINGS ---
TEST_SUITE_PATH = "/content/drive/MyDrive/SLM_DATA/test_suite_200_FINAL.jsonl"
MODEL_PATH = "/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed"

# --- 2. LOAD MODEL ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 2048,
    load_in_4bit = True
    # fix_mistral_regex = True # Removed, as it's not a valid argument for Qwen2 models
)
FastLanguageModel.for_inference(model)

# --- 3. SELECT RANDOM TEST CASE ---
with open(TEST_SUITE_PATH, 'r') as f:
    test_cases = [json.loads(line) for line in f]

sample = random.choice(test_cases)
instruction = sample.get("instruction", "")

print(f"📝 Testing on Random Sample from Test Suite...")
print(f"Problem Snippet: {instruction[:150]}...\n")

# --- 4. EXECUTE INFERENCE ---
prompt = f"### Instruction:\n{instruction}\n\n### Response:\n<tags>"
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 1500,
    temperature = 0.1,
    top_p = 0.9,
    use_cache = True
)

response = tokenizer.batch_decode(outputs)[0].split("### Response:\n")[-1]

print("🔥 STAGE 2 MODEL OUTPUT:")
print("-" * 40)
print(response)
print("-" * 40)

==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load /content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed as a legacy tokenizer.


📝 Testing on Random Sample from Test Suite...
Problem Snippet: ...

🔥 STAGE 2 MODEL OUTPUT:
----------------------------------------
<tags> EASY, MATH, GREEDY, OPTIMIZATION, GREEDY_ALGORITHMS </tags>
<think>
[DIFFICULTY]: EASY
[TOPICS]: MATH, GREEDY, OPTIMIZATION, GREEDY_ALGORITHMS
[GOAL]: Determine the minimum number of moves required to make all the coins in a row have the same value.
[LOGIC]: The problem can be solved by iteratively selecting the coin with the highest value and replacing it with the average of the remaining coins. This process continues until all coins have the same value.
[PSEUDOCODE]:
```python
def min_moves_to_equal_coins(coins):
    n = len(coins)
    if n == 1:
        return 0

    while True:
        max_coin = max(coins)
        min_coin = min(coins)
        if max_coin == min_coin:
            break

        # Replace the max_coin with the average of the remaining coins
        sum_remaining = sum(coins) - max_coin
        count_remaining = n - 1
        av

In [10]:
import json
import random
from unsloth import FastLanguageModel
from transformers import StoppingCriteria, StoppingCriteriaList
import torch

# --- 1. CONFIG & LOADING ---
MODEL_PATH = "/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed"
TEST_SUITE = "/content/drive/MyDrive/SLM_DATA/test_suite_200_FINAL.jsonl"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 2048, # Your updated max length
    load_in_4bit = True,
    # fix_mistral_regex = True
)
FastLanguageModel.for_inference(model)

# --- 2. STOPPING CRITERIA ---
class StopOnCodeEnd(StoppingCriteria):
    def __call__(self, input_ids, scores, **kwargs):
        decoded = tokenizer.decode(input_ids[0][-5:])
        return "</code>" in decoded

stop_criteria = StoppingCriteriaList([StopOnCodeEnd()])

# --- 3. PICK PROBLEM ---
with open(TEST_SUITE, 'r') as f:
    test_cases = [json.loads(line) for line in f]
    sample = random.choice(test_cases)
    problem_text = sample.get('problem', sample.get('instruction', ''))

print(f"📝 TEST SUITE PROBLEM:\n{'-'*40}\n{problem_text}\n{'-'*40}\n")

# --- 4. EXECUTION ---
prompt = f"""### Instruction:
Solve this competitive programming problem.
1. Use standard I/O (sys.stdin.read() or input()).
2. Output logic must be exactly: <tags>...</tags><think>...</think><code>...</code>.
3. NO example usage or hardcoded variables.

Problem: {problem_text}

### Response:
<tags>"""

inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 1500,
    temperature = 0.1,
    top_p = 0.9,
    stopping_criteria = stop_criteria,
    use_cache = True
)

# Extract only the model's response part
full_text = tokenizer.batch_decode(outputs)[0]
response = full_text.split("### Response:\n")[-1].strip()

print(f"🚀 MODEL SOLUTION:\n{'-'*40}\n{response}\n{'-'*40}")

==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load /content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed as a legacy tokenizer.


📝 TEST SUITE PROBLEM:
----------------------------------------
Solve the following coding problem using the programming language python:

Champa loved traveling the world. He loved going from one city to the other. Being the miser that he is, he never wishes spend any money. Champa, instead, jumps from one city to the other.  Also he likes trips of high quality.
He can start at any city of his choice. Given that he has visited the i^th city, he will not visit it again, he will only visit the remaining unvisited cities. This means that Champa will jump n - 1 times in total, so that he can visit all of the n cities.
Given two cities at heights, A and B, the amount of money spent is Q * |A - B| where Q is the quality of a trip.  Champa has a list of cities he's dying to visit. Can you tell the minimum amount of money Champa would have to spend to visit all of the cities?

Input
The first line contains T, the number of test cases.
Each test case is described by two lines.
The first line of

In [ ]:
import json
from tqdm import tqdm
from unsloth import FastLanguageModel
import torch
from transformers import StoppingCriteria, StoppingCriteriaList

# --- 1. CONFIG ---
TEST_SUITE = "/content/drive/MyDrive/SLM_DATA/test_suite_200_FINAL.jsonl"
MODEL_PATH = "/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed"
OUTPUT_FILE = "/content/drive/MyDrive/SLM_DATA/stage2_FINAL_EVAL_MATCHED.jsonl"

# --- 2. LOAD MODEL ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 2048,
    load_in_4bit = True,
    # fix_mistral_regex = True # Removed, as it's not a valid argument for Qwen2 models
)
FastLanguageModel.for_inference(model)

# --- 3. STOPPING CRITERIA (Prevents Double Vision) ---
class StopOnCodeEnd(StoppingCriteria):
    def __call__(self, input_ids, scores, **kwargs):
        decoded = tokenizer.decode(input_ids[0][-5:])
        return "</code>" in decoded

stop_criteria = StoppingCriteriaList([StopOnCodeEnd()])

# --- 4. RUN MATCHED EVALUATION ---
with open(TEST_SUITE, 'r') as f:
    test_cases = [json.loads(line) for line in f]

print(f"🚀 Starting Final Eval on {len(test_cases)} questions...")

with open(OUTPUT_FILE, 'w') as out_f:
    for case in tqdm(test_cases, desc="🤖 Generating"):
        # We extract the TRUE ID and TRUE PROBLEM from the ground truth
        true_id = case.get("id")
        problem_text = case.get("problem", case.get("instruction", ""))

        # Enhanced Prompting
        prompt = f"""### Instruction:
Solve this competitive programming problem.
1. Use standard I/O (sys.stdin.read() or input()).
2. Output logic must be exactly: <tags>...</tags><think>...</think><code>...</code>.
3. NO example usage or hardcoded variables.

Problem: {problem_text}

### Response:
<tags>"""

        inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens = 1500,
            temperature = 0.1,
            top_p = 0.9,
            stopping_criteria = stop_criteria,
            use_cache = True
        )

        full_response = tokenizer.batch_decode(outputs)[0].split("### Response:\n")[-1].strip()

        # Save with the TRUE ID and INSTRUCTION for the Judge
        result_entry = {
            "id": true_id,
            "instruction": problem_text,
            "prediction": full_response
        }
        out_f.write(json.dumps(result_entry) + "\n")

print(f"\n✅ Evaluation complete! Results saved to: {OUTPUT_FILE}")

==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load /content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed as a legacy tokenizer.


🚀 Starting Final Eval on 200 questions...


🤖 Generating:   0%|          | 0/200 [00:00<?, ?it/s]--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-